# CAAP — YOLO11n detection 5 class trên Kaggle

Upload ZIP đã chuẩn bị thành **Private Dataset**, import notebook này, **Add Input** dataset đó,
chọn **Accelerator → GPU** và bật **Internet** để cài phiên bản Ultralytics đã chốt.
Chạy các cell theo thứ tự. Notebook không cần clone repo.

Classes: `Accident, bus, car, motorcycle, truck`. Không dùng checkpoint segmentation cũ.
Dataset hiện là **DEVELOPMENT_ONLY_NOT_PAPER_EVALUATION**: đã cách ly lỗi box trùng,
nhưng review/nguồn chưa hoàn tất. Val dùng phát triển và chọn checkpoint; không có test độc lập.
Giữ Dataset/Notebook private vì ảnh và nhãn là dữ liệu nội bộ.

Notebook lưu dataset hash, phiên bản phần mềm, tham số và metric. Seed/deterministic hạn chế
biến thiên, không bảo đảm số giống tuyệt đối giữa GPU/phần mềm khác nhau.


In [ ]:
# Không cài lại torch: giữ bản CUDA của Kaggle.
import subprocess, sys
print('Installing Ultralytics 8.4.92; Internet must be enabled...', flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.92'])
print('Installation complete. Continue with configuration and dataset cells.', flush=True)


## 1. Cấu hình
Mặc định: YOLO11n, 640 px, 100 epochs tối đa, batch 16, early stopping 20 epochs.
Đây là baseline để thử, chưa bảo đảm hội tụ. Để kiểm tra nhanh, chọn `SMOKE = True` (1 epoch,
ảnh 320, batch 4; vẫn dùng split đầy đủ). Nếu hết VRAM, giảm BATCH rồi chạy lại bằng tên run mới.
Không tự hạ batch giữa một lượt để tránh thay đổi protocol âm thầm.


In [ ]:
from pathlib import Path
import os
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(WORK / 'ultralytics_config')
INPUT_ROOT = Path('/kaggle/input')
PACKAGE_ROOT = None  # Tự tìm; nếu gắn nhiều gói CAAP, điền Path('/kaggle/input/.../...')
SMOKE = False
SEED = 42
EPOCHS = 1 if SMOKE else 100
IMGSZ = 320 if SMOKE else 640
BATCH = 4 if SMOKE else 16
WORKERS = 2
RUN_NAME = 'caap_yolo11n_seed42_smoke' if SMOKE else 'caap_yolo11n_seed42'
RESUME_CHECKPOINT = ''  # last.pt từ một lượt bị ngắt, không phải best.pt đã train xong.
DATA_ROOT = WORK / 'caap_dataset'
DATA_YAML = WORK / 'caap_data.yaml'


## 2. Tìm gói, chép sang working và kiểm tra toàn bộ dữ liệu
Kaggle có thể giải nén ZIP khi tạo Dataset. Nếu ZIP vẫn nằm trong input, notebook sẽ tự giải nén
vào working. Không ghi cache/nhãn vào input. Hash sai, nhãn sai hoặc nhóm vượt split sẽ dừng.


In [ ]:
import csv, hashlib, importlib.util, json, shutil, zipfile

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

print('Locating CAAP dataset...', flush=True)
if PACKAGE_ROOT is None:
    candidates = sorted({p.parent for p in INPUT_ROOT.rglob('package_receipt.json')
                         if (p.parent / 'manifest.csv').is_file()})
    if not candidates:
        archives = sorted(INPUT_ROOT.rglob('caap_accident_vehicle_dev*.zip'))
        if len(archives) != 1:
            raise RuntimeError('Add Input đúng gói CAAP hoặc điền PACKAGE_ROOT; không tự chọn nhiều dataset.')
        extracted = WORK / 'caap_input_unzipped'
        if extracted.exists():
            raise RuntimeError('Đã có thư mục giải nén; điền PACKAGE_ROOT tới gói trong đó để dùng lại.')
        with zipfile.ZipFile(archives[0]) as z:
            for member in z.infolist():
                target = (extracted / member.filename).resolve()
                if extracted.resolve() not in target.parents:
                    raise ValueError('Unsafe ZIP path')
            print('Extracting ZIP; training has not started yet...', flush=True)
            z.extractall(extracted)
        candidates = sorted({p.parent for p in extracted.rglob('package_receipt.json')})
    if len(candidates) != 1:
        raise RuntimeError('Nhiều gói dữ liệu: điền PACKAGE_ROOT để chọn rõ ràng.')
    PACKAGE_ROOT = candidates[0]
PACKAGE_ROOT = Path(PACKAGE_ROOT).resolve()
print('Package found:', PACKAGE_ROOT, flush=True)
if DATA_ROOT.exists():
    if sha256(DATA_ROOT / 'package_receipt.json') != sha256(PACKAGE_ROOT / 'package_receipt.json'):
        raise RuntimeError('Working chứa snapshot khác; dùng session mới, không ghi đè.')
else:
    print('Copying dataset to working; training has not started yet...', flush=True)
    copied_files = [0]
    def copy_with_progress(source, destination):
        result = shutil.copy2(source, destination)
        copied_files[0] += 1
        if copied_files[0] == 1 or copied_files[0] % 500 == 0:
            print(f'Copied {copied_files[0]} files...', flush=True)
        return result
    shutil.copytree(PACKAGE_ROOT, DATA_ROOT, copy_function=copy_with_progress)
spec = importlib.util.spec_from_file_location('caap_checks', DATA_ROOT / 'kaggle_detection_helpers.py')
checks = importlib.util.module_from_spec(spec)
spec.loader.exec_module(checks)
print('Checking image hashes, labels and splits; training has not started yet...', flush=True)
original_digest = checks.digest
checked_images = [0]
expected_images = sum(v['images'] for v in json.loads((DATA_ROOT / 'report.json').read_text())['counts']['trial'].values())
def digest_with_progress(path):
    value = original_digest(path)
    if Path(path).parent.name == 'images':
        checked_images[0] += 1
        if checked_images[0] == 1 or checked_images[0] % 250 == 0:
            print(f'Checked {checked_images[0]}/{expected_images} images...', flush=True)
    return value
checks.digest = digest_with_progress
REPORT, MANIFEST = checks.verify_package(DATA_ROOT)
print(f'Dataset verification complete: {len(MANIFEST)} images. Continue to GPU check and Train cell.', flush=True)
DATASET_HASH = sha256(DATA_ROOT / 'manifest.csv')
print('Dataset:', PACKAGE_ROOT)
print('Mode:', REPORT['mode'])
print('Review snapshot:', REPORT['review_status'])
print(json.dumps(REPORT['counts']['trial'], indent=2, ensure_ascii=False))
print('Manifest SHA256:', DATASET_HASH)


## 3. Kiểm tra GPU, tạo YAML và xem nhãn
ID gốc: 0 Accident, 1 bus, 2 car, 3 motorcycle, 4 truck. Pipeline CAAP ánh xạ theo tên.
Xem overlay trước khi train: nếu phát hiện nhãn sai, sửa ở bộ review gốc rồi tạo snapshot mới.


In [ ]:
import platform, random
import numpy as np
import pandas as pd
import torch, ultralytics, yaml
from PIL import Image, ImageDraw
from IPython.display import display
if not torch.cuda.is_available():
    raise RuntimeError('Bật Accelerator GPU trên Kaggle trước khi train đầy đủ.')
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'ultralytics:', ultralytics.__version__)
assert ultralytics.__version__ == '8.4.92'
DATA_YAML.write_text(yaml.safe_dump({'path': str(DATA_ROOT), 'train': 'train/images',
    'val': 'val/images', 'nc': 5, 'names': REPORT['names']}, sort_keys=False))
display(pd.DataFrame(REPORT['counts']['trial']).T)
sample = random.Random(SEED).sample(MANIFEST, min(6, len(MANIFEST)))
canvas = Image.new('RGB', (960, 640), 'white')
for index, row in enumerate(sample):
    with Image.open(DATA_ROOT / row['image']) as original:
        im = original.convert('RGB').resize((320, 320))
    draw = ImageDraw.Draw(im)
    for line in (DATA_ROOT / row['label']).read_text().splitlines():
        if not line.strip(): continue
        cid, x, y, w, h = map(float, line.split())
        draw.rectangle(((x-w/2)*320, (y-h/2)*320, (x+w/2)*320, (y+h/2)*320), outline='lime', width=1)
        draw.text(((x-w/2)*320, (y-h/2)*320), REPORT['names'][int(cid)], fill='yellow')
    canvas.paste(im, ((index%3)*320, (index//3)*320))
display(canvas)


## 4. Train và lưu protocol
YOLO11n pretrained COCO được kèm gói. Chốt optimizer/augmentation rõ ràng, giữ validation
phát triển cố định, lưu checkpoint mỗi 5 epoch và best theo val. Không dùng test để chỉnh tham số.
Internet còn cần nếu Ultralytics kiểm tra AMP bằng checkpoint phụ.


In [ ]:
from datetime import datetime, timezone
from ultralytics import YOLO
RUN_DIR = WORK / 'runs' / RUN_NAME
TRAIN_ARGS = dict(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, workers=WORKERS, seed=SEED, deterministic=True, optimizer='AdamW',
    lr0=0.001, lrf=0.01, momentum=0.9, weight_decay=0.0005, cos_lr=True,
    warmup_epochs=3.0, patience=20, amp=True, cache=False, verbose=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, translate=0.1, scale=0.5,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0,
    close_mosaic=10, mixup=0.0, copy_paste=0.0, cutmix=0.0, multi_scale=0.0,
    val=True, plots=True, save=True, save_period=5,
    project=str(WORK / 'runs'), name=RUN_NAME, exist_ok=False)
PROTOCOL = {'mode': REPORT['mode'], 'dataset_manifest_sha256': DATASET_HASH,
    'package_receipt_sha256': sha256(DATA_ROOT / 'package_receipt.json'),
    'initial_checkpoint_sha256': sha256(DATA_ROOT / 'yolo11n.pt'),
    'class_names': REPORT['names'], 'seed': SEED, 'arguments': TRAIN_ARGS,
    'software': {'python': platform.python_version(), 'torch': torch.__version__,
                 'cuda': torch.version.cuda, 'ultralytics': ultralytics.__version__},
    'gpu': torch.cuda.get_device_name(0), 'started': datetime.now(timezone.utc).isoformat()}
receipt_path = WORK / (RUN_NAME + '_protocol.json')
if RESUME_CHECKPOINT:
    checkpoint = Path(RESUME_CHECKPOINT)
    previous_protocol = checkpoint.parent.parent / 'training_protocol.json'
    if not previous_protocol.is_file():
        raise RuntimeError('Resume cần training_protocol.json đi kèm thư mục weights.')
    previous = json.loads(previous_protocol.read_text())
    if previous['dataset_manifest_sha256'] != DATASET_HASH or previous['class_names'] != REPORT['names']:
        raise RuntimeError('Checkpoint thuộc snapshot/class khác.')
    checkpoint_info = torch.load(checkpoint, map_location='cpu', weights_only=False)
    if checkpoint_info.get('optimizer') is None or checkpoint_info.get('epoch', -1) < 0:
        raise RuntimeError('Checkpoint đã hoàn tất/stripped; dùng run mới để fine-tune, không resume.')
    PROTOCOL.update(arguments=previous['arguments'], resumed_from=str(checkpoint),
                    resumed_checkpoint_sha256=sha256(checkpoint))
    model = YOLO(str(checkpoint))
else:
    if RUN_DIR.exists():
        raise RuntimeError('Run đã tồn tại: đặt RUN_NAME mới hoặc chọn resume rõ ràng.')
    model = YOLO(str(DATA_ROOT / 'yolo11n.pt'))
assert model.task == 'detect'
PROTOCOL['status'] = 'RUNNING'
receipt_path.write_text(json.dumps(PROTOCOL, indent=2))
def save_protocol(trainer):
    (Path(trainer.save_dir) / 'training_protocol.json').write_text(json.dumps(PROTOCOL, indent=2))
model.add_callback('on_train_start', save_protocol)
model.add_callback('on_model_save', save_protocol)
print('Starting training now; model setup, AMP and label scan run before Epoch 1.', flush=True)
print('Epochs:', EPOCHS, 'batch:', BATCH, 'imgsz:', IMGSZ, 'output:', RUN_DIR, flush=True)
try:
    trained = model.train(resume=True, data=str(DATA_YAML), device=0, workers=WORKERS) if RESUME_CHECKPOINT else model.train(**TRAIN_ARGS)
    RUN_DIR = Path(model.trainer.save_dir)
    PROTOCOL.update(status='COMPLETED', finished=datetime.now(timezone.utc).isoformat(), output=str(RUN_DIR))
except BaseException as error:
    PROTOCOL.update(status='INTERRUPTED_OR_FAILED', error=str(error))
    raise
finally:
    receipt_path.write_text(json.dumps(PROTOCOL, indent=2))
    if getattr(model, 'trainer', None) is not None:
        save_protocol(model.trainer)
print('Run:', RUN_DIR)


## 5. Đánh giá best trên val phát triển, theo từng class
Không gọi val phát triển là test. Một lượt train chưa chứng minh chất lượng nhãn hoặc độ độc lập
nguồn. Chỉ thay detector trong CAAP sau khi kiểm tra riêng, giữ cùng checkpoint cho mọi ablation.


In [ ]:
best = RUN_DIR / 'weights' / 'best.pt'
assert best.is_file()
detector = YOLO(str(best))
assert detector.task == 'detect'
assert [detector.names[i] for i in range(5)] == REPORT['names']
validation = detector.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH,
    device=0, workers=WORKERS, plots=True, project=str(WORK / 'evaluation'),
    name=RUN_NAME + '_dev_val', exist_ok=False)
rows = []
for metric_index, class_id in enumerate(validation.box.ap_class_index):
    rows.append({'class_id': int(class_id), 'class': REPORT['names'][int(class_id)],
        'precision': float(validation.box.p[metric_index]), 'recall': float(validation.box.r[metric_index]),
        'mAP50': float(validation.box.ap50[metric_index]), 'mAP50_95': float(validation.box.ap[metric_index])})
per_class = pd.DataFrame(rows)
per_class.to_csv(RUN_DIR / 'development_per_class.csv', index=False)
display(per_class)
summary = {'mode': REPORT['mode'], 'split': 'development_val', 'independent_test': False,
    'dataset_manifest_sha256': DATASET_HASH, 'best_checkpoint_sha256': sha256(best),
    'metrics': {key: float(value) for key, value in validation.results_dict.items()},
    'per_class': rows, 'limitations': REPORT['limitations']}
(RUN_DIR / 'development_metrics.json').write_text(json.dumps(summary, indent=2))
display(pd.read_csv(RUN_DIR / 'results.csv').tail())


## 6. Xem dự đoán và tải kết quả
Ảnh dự đoán/figure cũng là dữ liệu nội bộ: giữ Notebook private. Output gồm best.pt, last.pt,
args.yaml, results.csv, protocol, metric theo class và biểu đồ. Save Version để giữ output;
Kaggle có thể ngắt session, tải checkpoint về sau lượt chạy. File periodic có thể dùng resume
khi đi kèm training_protocol.json và đúng snapshot.


In [ ]:
val_sample = random.Random(SEED).sample([r for r in MANIFEST if r['split']=='val'],
    min(8, sum(r['split']=='val' for r in MANIFEST)))
predictions = detector.predict(source=[str(DATA_ROOT / r['image']) for r in val_sample],
    imgsz=IMGSZ, conf=0.25, device=0, save=True, project=str(WORK / 'predictions'),
    name=RUN_NAME, exist_ok=False, verbose=False)
for prediction in predictions[:4]:
    display(Image.fromarray(prediction.plot()[..., ::-1]))
for name in ['manifest.csv', 'report.json', 'package_receipt.json']:
    shutil.copy2(DATA_ROOT / name, RUN_DIR / name)
archive = shutil.make_archive(str(WORK / (RUN_NAME + '_outputs')), 'zip', root_dir=RUN_DIR)
from IPython.display import FileLink
display(FileLink(archive))
print('Checkpoint dùng tích hợp sau khi kiểm tra:', best)
print('Class names:', detector.names)
print('Kết quả hiện chỉ dùng development; chưa báo cho paper.')


Tài liệu chính thức: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks),
[Ultralytics Train](https://docs.ultralytics.com/modes/train).
